In [197]:
import sympy as sp
import numpy as np
import pyshtools
import matplotlib.pyplot as plt

# 定义变量
theta, phi = sp.symbols('theta phi', real=True)

# 定义函数 f(θ, φ)
f = sp.sin(theta) * sp.cos(phi)

# 一阶偏导数
df_dtheta = sp.diff(f, theta)
df_dphi = sp.diff(f, phi)

# 球面梯度
grad_theta = df_dtheta                   # e_theta 分量
grad_phi = df_dphi / sp.sin(theta)      # e_phi 分量

# 显示梯度
print("球面梯度 ∇ₛf:")
print("θ 分量:", grad_theta)
print("φ 分量:", grad_phi)

# Laplace-Beltrami 算子
laplacian = (1 / sp.sin(theta)) * sp.diff(sp.sin(theta) * df_dtheta, theta) \
            + (1 / sp.sin(theta)**2) * sp.diff(df_dphi, phi)

# 显示拉普拉斯
print("\n球面拉普拉斯 Δₛf:")
sp.pprint(sp.simplify(laplacian))

球面梯度 ∇ₛf:
θ 分量: cos(phi)*cos(theta)
φ 分量: -sin(phi)

球面拉普拉斯 Δₛf:
-2⋅sin(θ)⋅cos(φ)


In [198]:
# 设置最大阶数和网格
lmax = 15
clm = pyshtools.SHCoeffs.from_random([0])
grid = clm.expand(grid='DH', lmax=lmax, extend=False)
lat_m, lon_m = np.meshgrid(np.radians(grid.lats()), np.radians(grid.lons()), indexing='ij')

# 定义函数 f = cos(lat) * cos(lon)
f_vals = np.cos(lat_m) * np.cos(lon_m)

# 转为 SHGrid 并计算拉普拉斯
f_grid = pyshtools.SHGrid.from_array(f_vals)
f_coeffs = f_grid.expand()

# 拉普拉斯系数处理：乘以 -l(l+1)
laplacian_coeffs = f_coeffs.copy()

# 构造 -l(l+1) 因子
factors = -np.arange(lmax + 1) * (np.arange(lmax + 1) + 1)  # shape: (lmax+1,)

# 广播乘法：对所有 m 应用该因子
laplacian_coeffs.coeffs[0, :, :] *= factors[:, np.newaxis]
laplacian_coeffs.coeffs[1, :, :] *= factors[:, np.newaxis]

# 展开回网格
laplacian_grid = laplacian_coeffs.expand(grid='DH', extend=False)

# 分析解：-2 * cos(lat) * cos(lon)
analytical = -2 * np.cos(lat_m) * np.cos(lon_m)
error = laplacian_grid.data - analytical
print("最大误差：", np.max(np.abs(error)))

最大误差： 2.9332092310596636e-13


In [199]:
# 定义函数 f = cos(lat) * cos(lon)
f_vals = np.cos(lat_m) * np.cos(lon_m)

# 转为 SHGrid 并计算拉普拉斯
f_grid = pyshtools.SHGrid.from_array(f_vals)
f_coeffs = f_grid.expand()

# 梯度计算验证
df = f_coeffs.gradient(grid='DH', extend=False)

# 理论导数
grad_theta_analytical = -np.sin(lat_m) * np.cos(lon_m)
grad_phi_analytical = -np.cos(lat_m) * np.sin(lon_m)

# 误差分析
theta_error = df.theta.data[1:,:]*(-1) - grad_theta_analytical[1:,:]  # d/dtheta = -d/dlat
phi_error = df.phi.data * np.sin(np.pi / 2 - lat_m) - grad_phi_analytical   # d/dphi = d/dlon*sin(theta)

# 显示误差
print("θ分量最大误差:", np.max(np.abs(theta_error)))
print("φ分量最大误差:", np.max(np.abs(phi_error)))

θ分量最大误差: 2.0650148258027912e-14
φ分量最大误差: 3.774758283725532e-15


In [200]:
import numpy as np
import shtns
import matplotlib.pyplot as plt

# 初始化 shtns 对象
lmax = 15
sht = shtns.sht(lmax)
nlats, nlons = sht.set_grid()  # 自动选择合适网格
print("网格大小:", nlats, nlons)
lats = np.arcsin(sht.cos_theta)
lons = (2.*np.pi/nlons)*np.arange(nlons)
theta_m, phi_m = np.meshgrid(lats, lons, indexing='ij')

# 定义函数 f(θ, φ) = cos(θ) * cos(φ)
f = np.cos(theta_m) * np.cos(phi_m)

# 转换到频域（球谐系数）
f_lm = sht.analys(f)

# 实际要计算一阶导数 ∂f/∂θ
# 使用 shtns 的 grd_theta 和 grd_phi 工具
df_theta, df_phi = sht.synth_grad(f_lm) 

# 解析解
grad_theta_analytical = -np.sin(theta_m) * np.cos(phi_m)
grad_phi_analytical = -np.cos(theta_m) * np.sin(phi_m)

# 分析误差（注意经纬转换）
theta_error = df_theta * (-1) - grad_theta_analytical
phi_error = df_phi * np.sin(np.pi / 2 - theta_m) - grad_phi_analytical

print("θ分量最大误差:", np.max(np.abs(theta_error)))
print("φ分量最大误差:", np.max(np.abs(phi_error)))


网格大小: 16 32
θ分量最大误差: 1.587618925213974e-14
φ分量最大误差: 3.6637359812630166e-15
